# Đánh giá + render demo (chạy trên Kaggle, không chạy trên máy local)
**Cách dùng:** sửa code ở máy → `git push` → mở notebook này → **Save Version → Save & Run All**. Kết quả ở tab **Output**.

Cần trước:
- Settings: **GPU** (T4 / P100) + **Internet: On**
- **Add Input** → Kaggle Dataset chứa `best.pt` (detector) và `pitch_best.pt` (keypoint sân)

In [ ]:
WEIGHTS_DIR = "/kaggle/input/football-weights"  # đổi nếu dataset của bạn tên khác
GIT_REF = "main"  # hoặc 1 commit / tag cũ (vd "423f64e", "v1.0-radar") để chạy lại đúng phiên bản đó

In [ ]:
!git clone -q https://github.com/thaisanh1602/football_cv_tracking.git
%cd football_cv_tracking
!git checkout -q {GIT_REF} && git log --oneline -1   # ghi lại đang chạy phiên bản nào
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
DET = f"{WEIGHTS_DIR}/best.pt"
PITCH = f"{WEIGHTS_DIR}/pitch_best.pt"
for p in (DET, PITCH):
    print(p, "OK" if Path(p).exists() else "KHÔNG THẤY -> kiểm tra Add Input / WEIGHTS_DIR")
OUT = "/kaggle/working/results"
!mkdir -p {OUT}

## A. Radar: sai số homography (m), độ rung, tốc độ / quãng đường trên SoccerNet GSR test

In [ ]:
GSR = "/kaggle/working/gsr_test"
!python scripts/download_gsr.py --split test --clips SNGS-116 SNGS-117 SNGS-118 --out {GSR}
!python scripts/eval_pitch.py --data-root {GSR} --pitch-weights {PITCH} --out {OUT}/pitch_eval
!python scripts/eval_kinematics.py --data-root {GSR} --pitch-weights {PITCH} --kpt-cache {OUT}/pitch_eval/kpt_cache --out {OUT}/kinematics_eval

## A2. Nhấp nháy mask (YOLO-seg) trên GSR test: mask gốc vs ổn định theo track


In [ ]:
!python scripts/eval_mask_flicker.py --data-root {GSR} --seg-weights yolo11m-seg.pt --cache /kaggle/working/mask_cache --out {OUT}/mask_eval
!python scripts/eval_radar_filter.py --data-root {GSR} --pitch-weights {PITCH} --kpt-cache {OUT}/pitch_eval/kpt_cache --out {OUT}/radar_filter_eval
!rm -rf /kaggle/working/mask_cache


## B. Demo: radar + phân đội + tốc độ + bóng người, và video so sánh trước / sau chống rung


In [ ]:
CLIP = "SNGS-118"  # đoạn 15–25 s có radar phủ ~98%
VID = f"/kaggle/working/{CLIP}.mp4"
!ffmpeg -y -loglevel error -framerate 25 -i {GSR}/{CLIP}/img1/%06d.jpg -c:v libx264 -pix_fmt yuv420p -crf 18 {VID}

COMMON = f"--source {VID} --weights {DET} --pitch-weights {PITCH} --team --seg"
# SAU: mặc định (H làm mượt + Kalman theo cầu thủ trên radar + mask ổn định α=0.5)
!python scripts/infer_video.py {COMMON} --output /kaggle/working/after.mp4
# TRƯỚC: H thô, không lọc radar, mask gốc từng frame
!python scripts/infer_video.py {COMMON} --no-smooth --no-radar-filter --mask-alpha 1.0 --output /kaggle/working/before.mp4

!python scripts/side_by_side.py --left /kaggle/working/before.mp4 --right /kaggle/working/after.mp4 --output /kaggle/working/compare_raw.mp4     --left-note "raw homography, no radar filter, raw masks" --right-note "smoothed H + per-player Kalman (radar jitter -90%) + stabilized masks"

# H.264 để xem trên trình duyệt / YouTube, GIF ngắn cho README
!ffmpeg -y -loglevel error -i /kaggle/working/after.mp4 -c:v libx264 -crf 23 -pix_fmt yuv420p {OUT}/radar_demo.mp4
!ffmpeg -y -loglevel error -i /kaggle/working/compare_raw.mp4 -c:v libx264 -crf 23 -pix_fmt yuv420p {OUT}/compare.mp4
!ffmpeg -y -loglevel error -ss 15 -t 6 -i {OUT}/compare.mp4 -vf "fps=8,scale=960:-1:flags=lanczos,split[a][b];[a]palettegen=max_colors=96[p];[b][p]paletteuse=dither=bayer:bayer_scale=4" {OUT}/compare.gif
!rm /kaggle/working/before.mp4 /kaggle/working/after.mp4 /kaggle/working/compare_raw.mp4 {VID}
!ls -la {OUT}


## C. (Tuỳ chọn) Tracking: ByteTrack vs BoT-SORT trên SoccerNet-Tracking test, 10 sequence

In [ ]:
RUN_TRACKING = False  # bật khi cần cập nhật bảng tracking (tải ~10 GB, lâu)
if RUN_TRACKING:
    from SoccerNet.Downloader import SoccerNetDownloader
    SoccerNetDownloader(LocalDirectory="/kaggle/working/SoccerNet").downloadDataTask(task="tracking", split=["test"])
    !cd /kaggle/working/SoccerNet/tracking && unzip -q test.zip && rm test.zip
    TRACKERS = "configs/bytetrack.yaml configs/botsort.yaml configs/botsort_tuned.yaml"
    !python scripts/eval_soccernet.py --data-root /kaggle/working/SoccerNet/tracking/test --weights {DET} --limit 10 --out {OUT}/trackeval --cache /kaggle/working/det_cache --trackers {TRACKERS}
    !rm -rf /kaggle/working/SoccerNet /kaggle/working/det_cache

## Tóm tắt số liệu (copy vào README)

In [ ]:
for f in sorted(Path(OUT).rglob("summary.json")):
    print("==", f.relative_to(OUT))
    print(f.read_text()[:2000])


In [ ]:
# Dọn dữ liệu tải về để tab Output chỉ còn kết quả (Kaggle giới hạn 20 GB output)
!rm -rf {GSR} {OUT}/pitch_eval/kpt_cache